In [15]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# Load the labeled data CSV
df = pd.read_csv('csvFiles/fine-tuned-data-conversation-rate.csv')

# Split the data into training and testing
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

# # Tokenize the queries and answers
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# # Encode the data
train_encodings = tokenizer(train_df['Question'].tolist(), truncation=True, padding=True)
test_encodings = tokenizer(test_df['Question'].tolist(), truncation=True, padding=True)

# Convert the answers to the format needed by the model
# train_labels = [answer_to_label(answer) for answer in train_df['answer']]
# test_labels = [answer_to_label(answer) for answer in test_df['answer']]

In [16]:
print('train_df',train_df)
print('test_df',test_df)
print('train_encodings',train_encodings)
print('test_encodings',test_encodings)

train_df                                             Question  \
1  Can you share a list of channels where convers...   
4  Can you share a list of content where conversi...   
2  Can you share a list of channels where convers...   
0  Can you share a list of channels where convers...   

                                              Answer  
1  Channel: BEONDTV  Platform: Beond.TV  Delivery...  
4  BEONDTV Author Series, Carlos  Lisa Episode 15...  
2  Channel: BEONDTV  Platform: LG  Delivery Regio...  
0  Channel: BEONDTV  Platform: LG  Delivery Regio...  
test_df                                             Question  \
3  Can you share a list of channels where convers...   

                                              Answer  
3  Channel: BEONDTV  Platform: Beond.TV  Delivery...  
train_encodings {'input_ids': [[101, 2064, 2017, 3745, 1037, 2862, 1997, 6833, 2073, 7584, 3446, 2003, 2625, 2084, 2184, 1003, 102, 0, 0, 0, 0, 0], [101, 2064, 2017, 3745, 1037, 2862, 1997, 4180, 2073, 75

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

# Load the model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encodings,
    eval_dataset=test_encodings,
)


In [5]:
# from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

from datasets import Dataset
import pandas as pd
# Load and prepare the data
df = pd.read_csv('csvFiles/fine-tuned-data-conversation-rate.csv')
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Initialize tokenizer and model
model_name = "sentence-transformers/all-mpnet-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=df['label'].nunique())


# Tokenize the data
def tokenize_function(examples):
    return tokenizer(examples['Question'], padding='max_length', truncation=True)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Initialize the model
# model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(df['label'].unique()))

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
)

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()

eval_results = trainer.evaluate(eval_dataset=test_dataset)
print('eval_results',eval_results)

model.save_pretrained('./fine-tuned-models/all-mpnet-base-v2')
tokenizer.save_pretrained('./fine-tuned-models/all-mpnet-base-v2')



Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 3/3 [00:01<00:00,  1.64it/s]


{'train_runtime': 1.8342, 'train_samples_per_second': 6.542, 'train_steps_per_second': 1.636, 'train_loss': 1.6045811971028645, 'epoch': 3.0}


100%|██████████| 1/1 [00:00<00:00, 172.68it/s]


eval_results {'eval_loss': 1.605398416519165, 'eval_runtime': 0.1292, 'eval_samples_per_second': 7.741, 'eval_steps_per_second': 7.741, 'epoch': 3.0}


('./fine-tuned-models/all-mpnet-base-v2/tokenizer_config.json',
 './fine-tuned-models/all-mpnet-base-v2/special_tokens_map.json',
 './fine-tuned-models/all-mpnet-base-v2/vocab.txt',
 './fine-tuned-models/all-mpnet-base-v2/added_tokens.json',
 './fine-tuned-models/all-mpnet-base-v2/tokenizer.json')

In [35]:
# using the model

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the saved model and tokenizer
model_name = './fine-tuned-models/all-mpnet-base-v2'
tokenizer_name = './fine-tuned-models/all-mpnet-base-v2'

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = logits.argmax().item()
    return predicted_class

text = "Can you share a list of content where conversion rate is greater than 10% but less than 50%"
predicted_label = predict(text)
print(f'Predicted label: {predicted_label}')


Predicted label: 3
